# Model 3（全量）：MaSE Phase2.5+TTA ∥ RF 概率融合

5% 版 `Model3_Phase25_TTA_RF_Blend.ipynb` 的 **full DeepYeast** 对应实现。  
另一台机器上 **不会有 5% 的 RF**，因此本 notebook **自带 Model 2**（冻 Phase2.5 → logits 特征 + TTA 平均 → RF），再做融合。

$$
p = \alpha\, p_{\mathrm{MaSE\text{-}TTA}} + (1-\alpha)\, p_{\mathrm{RF}}
$$

| Stage | 做什么 |
|-------|--------|
| 0 | 冻住全量 v6 Phase 2.5 `best.pt` |
| 1 | 抽 train/val/test 的 61 维 stack logits（TTA×7 特征均值） |
| 2 | RF 网格 `n_estimators ∈ {100,200,300,1000}`，val 选优 |
| 3 | val/test 上算 MaSE TTA 软投票概率 |
| 4 | val 扫 α；并列取 plateau **中位 α**；只在 test 上报一次 |

**全量对照（已有结果，seed=42）**

| 模型 | Full test |
|------|-----------|
| v6 Phase 2.5 | ~89.07% |
| v6 Phase 2.5 + TTA | **~89.82%** |
| RF / Blend | 本 notebook 写出 |

**前提（另一台电脑）**

1. 全量数据 `deepyeast_full`（90,000 图）。见 `FULL_DATA_QUICKSTART.md`。
2. 已训好的 `mase_lite_full_v6_phase25/best.pt`（见 `V6_PHASE25_RUN.md`）。
3. GPU 强烈建议：TTA 特征提取要对 train 65k × 7 视图前向，CPU 会极慢。

无界面机器可改跑仓库根目录脚本：

```bash
python run_model3_full.py --data-dir /path/to/deepyeast_full --device cuda
```

## 1. 路径（按本机改这一格）

在 **仓库根目录** 打开本 notebook，或设置环境变量 `MASE_REPO` / `MASE_DATA`。

In [ ]:
from pathlib import Path
import os
import sys

REPO = Path(os.environ.get("MASE_REPO", Path.cwd())).resolve()
if not (REPO / "pytorch").exists():
    raise SystemExit(
        f"找不到 pytorch/ ：请把 notebook 的工作目录设为仓库根，或 export MASE_REPO=...\n当前 REPO={REPO}"
    )

DATA_DIR = Path(os.environ.get("MASE_DATA", REPO / "deepyeast_full")).resolve()
PHASE25_DIR = Path(
    os.environ.get(
        "MASE_PHASE25_DIR",
        DATA_DIR / "checkpoints" / "mase_lite_full_v6_phase25",
    )
).resolve()
PHASE25_CKPT = PHASE25_DIR / "best.pt"
PHASE25_RESULTS = PHASE25_DIR / "results.json"
TTA_SUMMARY = PHASE25_DIR / "tta_eval" / "summary.json"
OUT_DIR = DATA_DIR / "checkpoints" / "mase_model3_phase25_tta_rf_blend_full"
MODEL_2_DIR = OUT_DIR / "model2_rf"

SEED = 42
FEATURE_MODE = "logits"
USE_TTA_FEATURES = True
TTA_MODE = "flip_rot"
BATCH_SIZE = 64
NUM_WORKERS = 4 if os.name != "nt" else 0
DEVICE_PREF = "auto"  # auto|cuda|mps|cpu
ALPHAS = [i / 20 for i in range(21)]

sys.path.insert(0, str(REPO / "pytorch"))
OUT_DIR.mkdir(parents=True, exist_ok=True)
MODEL_2_DIR.mkdir(parents=True, exist_ok=True)

print("REPO        :", REPO)
print("DATA_DIR    :", DATA_DIR, "exists=", DATA_DIR.exists())
print("PHASE25_CKPT:", PHASE25_CKPT, "exists=", PHASE25_CKPT.exists())
print("OUT_DIR     :", OUT_DIR)
if PHASE25_CKPT.exists() and PHASE25_CKPT.stat().st_size < 100_000:
    print("⚠ checkpoint 过小，可能是 Git LFS 指针 → git lfs pull")

## 2. 加载 Phase 2.5

In [ ]:
import json
import time
import numpy as np
import joblib
import torch
from sklearn.ensemble import RandomForestClassifier

from dataset import make_loaders
from eval_mase_tta import forward_tta, tta_views
from eval_mase_uq import config_from_results_json, load_checkpoint, resolve_device, set_seed
from mase_lite_net import MASE_LITE_DEFAULT, MaSELiteNet
from mase_trad_net import build_stack_features

assert PHASE25_CKPT.exists(), f"缺少 Phase2.5: {PHASE25_CKPT}"
assert DATA_DIR.exists() and (DATA_DIR / "train").exists(), f"缺少全量数据: {DATA_DIR}"

set_seed(SEED)
device = resolve_device(DEVICE_PREF)
print("device=", device)

cfg = config_from_results_json(PHASE25_RESULTS if PHASE25_RESULTS.exists() else None) or MASE_LITE_DEFAULT
model = MaSELiteNet(cfg).to(device).eval()
load_checkpoint(model, PHASE25_CKPT)
print("loaded", PHASE25_CKPT)

train_loader, val_loader, test_loader = make_loaders(
    DATA_DIR,
    batch_size=BATCH_SIZE,
    augment=False,
    strong_augment=False,
    num_workers=NUM_WORKERS,
)

## 3. Model 2：TTA 特征 → RF

已有 `model2_rf/features_*.npz` 与 `rf_best.joblib` 时会跳过，可断点续跑。

In [ ]:
@torch.inference_mode()
def extract_split(model, loader, device, *, feature_mode: str, use_tta: bool):
    xs, ys = [], []
    n, t0 = 0, time.time()
    for images, labels in loader:
        images = images.to(device)
        if use_tta:
            view_feats = []
            for view in tta_views(images, "flip_rot"):
                _, details = model(view, train=False, return_details=True)
                f, _ = build_stack_features(details, mode=feature_mode, use_disagreement=True)
                view_feats.append(f)
            feats = torch.stack(view_feats, dim=0).mean(dim=0)
        else:
            _, details = model(images, train=False, return_details=True)
            feats, _ = build_stack_features(details, mode=feature_mode, use_disagreement=True)
        xs.append(feats.cpu().numpy())
        ys.append(labels.numpy())
        n += 1
        if n % 20 == 0:
            print(f"    batches={n}  elapsed={time.time()-t0:.0f}s", flush=True)
    return np.concatenate(xs, 0).astype(np.float64), np.concatenate(ys, 0).astype(np.int64)


feat_paths = {s: MODEL_2_DIR / f"features_{s}.npz" for s in ("train", "val", "test")}
if all(p.exists() for p in feat_paths.values()):
    print("reuse Model 2 features")
else:
    t_feat = time.time()
    for name, loader in [("train", train_loader), ("val", val_loader), ("test", test_loader)]:
        print(f"extracting {name} ...", flush=True)
        X, y = extract_split(
            model, loader, device, feature_mode=FEATURE_MODE, use_tta=USE_TTA_FEATURES
        )
        np.savez_compressed(feat_paths[name], X=X, y=y)
        print(f"  {name}: X={X.shape} y={y.shape}")
    with (MODEL_2_DIR / "feature_meta.json").open("w") as f:
        json.dump(
            {
                "feature_mode": FEATURE_MODE,
                "use_tta_features": USE_TTA_FEATURES,
                "tta_mode": TTA_MODE if USE_TTA_FEATURES else None,
                "checkpoint": str(PHASE25_CKPT),
                "seed": SEED,
            },
            f,
            indent=2,
        )
    print(f"features done in {(time.time()-t_feat)/60:.1f} min")

rf_path = MODEL_2_DIR / "rf_best.joblib"
model2_results_path = MODEL_2_DIR / "results.json"
if rf_path.exists() and model2_results_path.exists():
    rf = joblib.load(rf_path)
    model2_summary = json.load(open(model2_results_path))
    print("reuse RF", rf_path)
else:
    tr, va, te = (np.load(feat_paths[s]) for s in ("train", "val", "test"))
    Xtr, ytr = tr["X"], tr["y"]
    Xva, yva = va["X"], va["y"]
    Xte, yte = te["X"], te["y"]
    rows, best = [], None
    for n in [100, 200, 300, 1000]:
        clf = RandomForestClassifier(n_estimators=n, n_jobs=-1, random_state=SEED)
        t0 = time.time()
        clf.fit(Xtr, ytr)
        val_acc = float((clf.predict(Xva) == yva).mean())
        test_acc = float((clf.predict(Xte) == yte).mean())
        row = {"n_estimators": n, "val_accuracy": val_acc, "test_accuracy": test_acc}
        rows.append(row)
        print(f"RF n={n:4d}  val={val_acc:.4f}  test={test_acc:.4f}  ({time.time()-t0:.0f}s)")
        if best is None or val_acc > best["val_accuracy"]:
            best = {**row, "clf": clf}
    joblib.dump(best["clf"], rf_path)
    model2_summary = {
        "method": "mase_model2_phase25_tta_rf_full",
        "backbone": str(PHASE25_CKPT),
        "feature_mode": FEATURE_MODE,
        "use_tta_features": USE_TTA_FEATURES,
        "seed": SEED,
        "grid": rows,
        "best_n_estimators": best["n_estimators"],
        "best_val_accuracy": best["val_accuracy"],
        "test_accuracy": best["test_accuracy"],
    }
    with model2_results_path.open("w") as f:
        json.dump(model2_summary, f, indent=2)
    rf = best["clf"]
    print(
        f"Model 2 best n={best['n_estimators']}  "
        f"val={best['val_accuracy']:.4f}  test={best['test_accuracy']:.4f}"
    )

## 4. MaSE TTA 概率 + 与 RF 对齐

In [ ]:
def acc(p, y):
    return float((p.argmax(axis=1) == y).mean())


def align_rf_proba(rf, p_rf):
    if hasattr(rf, "classes_") and not np.array_equal(rf.classes_, np.arange(len(rf.classes_))):
        mapped = np.zeros_like(p_rf)
        for j, c in enumerate(rf.classes_):
            mapped[:, int(c)] = p_rf[:, j]
        return mapped
    return p_rf


@torch.inference_mode()
def collect_mase_tta_probs(model, loader, device, tta_mode: str):
    ps, ys = [], []
    n, t0 = 0, time.time()
    for images, labels in loader:
        images = images.to(device)
        p = forward_tta(model, images, tta_mode).cpu().numpy()
        ps.append(p)
        ys.append(labels.numpy())
        n += 1
        if n % 20 == 0:
            print(f"    batches={n}  elapsed={time.time()-t0:.0f}s", flush=True)
    return np.concatenate(ps, 0).astype(np.float64), np.concatenate(ys, 0).astype(np.int64)


packs = {}
for name, loader in [("val", val_loader), ("test", test_loader)]:
    prob_path = OUT_DIR / f"probs_{name}.npz"
    if prob_path.exists():
        z = np.load(prob_path)
        packs[name] = {"p_mase": z["p_mase"], "p_rf": z["p_rf"], "y": z["y"]}
        print("reuse", prob_path.name)
        continue
    print(f"MaSE TTA → {name} ...", flush=True)
    p_mase, y = collect_mase_tta_probs(model, loader, device, TTA_MODE)
    feat = np.load(feat_paths[name])
    assert np.array_equal(feat["y"], y), f"{name}: 特征与 TTA 标签顺序不一致"
    p_rf = align_rf_proba(rf, rf.predict_proba(feat["X"]).astype(np.float64))
    packs[name] = {"p_mase": p_mase, "p_rf": p_rf, "y": y}
    np.savez_compressed(prob_path, p_mase=p_mase, p_rf=p_rf, y=y)
    print(f"  {name}: n={len(y)}  MaSE-TTA={acc(p_mase, y):.4f}  RF={acc(p_rf, y):.4f}")

## 5. Val 扫 α → Test 融合

In [ ]:
rows = []
for a in ALPHAS:
    p_va = a * packs["val"]["p_mase"] + (1.0 - a) * packs["val"]["p_rf"]
    p_te = a * packs["test"]["p_mase"] + (1.0 - a) * packs["test"]["p_rf"]
    row = {
        "alpha": a,
        "val_accuracy": acc(p_va, packs["val"]["y"]),
        "test_accuracy": acc(p_te, packs["test"]["y"]),
    }
    rows.append(row)
    print(f"α={a:.2f}  val={row['val_accuracy']:.4f}  test={row['test_accuracy']:.4f}")

best_val = max(r["val_accuracy"] for r in rows)
plateau = [r for r in rows if r["val_accuracy"] == best_val]
best = plateau[len(plateau) // 2]
print(f"val plateau={[r['alpha'] for r in plateau]} → median α={best['alpha']:.2f}")

mase_tta_test = acc(packs["test"]["p_mase"], packs["test"]["y"])
rf_test = acc(packs["test"]["p_rf"], packs["test"]["y"])
summary = {
    "method": "mase_model3_phase25_tta_rf_blend_full",
    "formula": "p = alpha * p_MaSE_TTA + (1-alpha) * p_RF",
    "selection_rule": "max val_accuracy; ties → median alpha in plateau",
    "data_dir": str(DATA_DIR),
    "backbone": str(PHASE25_CKPT),
    "rf_path": str(rf_path),
    "tta_mode": TTA_MODE,
    "feature_mode": FEATURE_MODE,
    "use_tta_features": USE_TTA_FEATURES,
    "seed": SEED,
    "alphas": ALPHAS,
    "grid": rows,
    "val_plateau_alphas": [r["alpha"] for r in plateau],
    "best_alpha": best["alpha"],
    "best_val_accuracy": best["val_accuracy"],
    "test_accuracy": best["test_accuracy"],
    "mase_tta_test": mase_tta_test,
    "rf_test": rf_test,
    "delta_vs_mase_tta_pp": 100.0 * (best["test_accuracy"] - mase_tta_test),
    "delta_vs_rf_pp": 100.0 * (best["test_accuracy"] - rf_test),
    "model2": {
        "best_n_estimators": model2_summary.get("best_n_estimators"),
        "test_accuracy": model2_summary.get("test_accuracy"),
    },
}
if TTA_SUMMARY.exists():
    summary["mase_phase25_tta_test_reported"] = json.load(open(TTA_SUMMARY)).get("test_tta_accuracy")

with (OUT_DIR / "results.json").open("w") as f:
    json.dump(summary, f, indent=2)

print("\n=== Model 3 full (val-selected α) ===")
print(f"best α={best['alpha']:.2f}  val={best['val_accuracy']:.4f}  test={best['test_accuracy']:.4f}")
print(f"MaSE+TTA test={mase_tta_test:.4f}  Δ={summary['delta_vs_mase_tta_pp']:+.2f} pp")
print(f"RF       test={rf_test:.4f}  Δ={summary['delta_vs_rf_pp']:+.2f} pp")
print("saved:", OUT_DIR / "results.json")

## 6. 读结果口径

```text
deepyeast_full/checkpoints/mase_model3_phase25_tta_rf_blend_full/
  model2_rf/
    features_{train,val,test}.npz
    rf_best.joblib
    results.json
  probs_{val,test}.npz
  results.json          ← 融合主结果：test_accuracy / best_alpha
```

| 模型 | Full test |
|------|-----------|
| Phase2.5 + TTA | `mase_tta_test`（对照约 **89.82%**） |
| RF | `rf_test` |
| Blend | `test_accuracy` @ `best_alpha` |

命令行等价：`python run_model3_full.py --data-dir /path/to/deepyeast_full --device cuda`